In [1]:
import tensorflow as tf
tf.config.experimental.set_visible_devices([], "GPU") # Отключение GPU для TensorFlow

import os
import jax
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.8'

from td_sa_stack import get_dataset, TrainerModuleSingle, RegressionInceptionNetV1

%load_ext autoreload
%autoreload 2

2025-06-14 14:11:03.088038: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749910263.108112   31769 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749910263.114545   31769 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749910263.129670   31769 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1749910263.129686   31769 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1749910263.129688   31769 computation_placer.cc:177] computation placer alr

In [2]:
import sys
from absl import flags
from ml_collections.config_flags import config_flags

sys.argv = [
    "",
    "--config=td_sa_stack/config.py",
]

config_flags.DEFINE_config_file("config", None, "Training configuration.", lock_config=True)
# flags.DEFINE_string("workdir", None, "Work directory.")
# flags.DEFINE_enum("mode", None, ["train", "eval", "fid_stats"], "Running mode: train, eval or fid_stats")
# flags.DEFINE_string("eval_folder", "eval", "The folder name for storing evaluation results")


FLAGS = flags.FLAGS
FLAGS(sys.argv)

config = FLAGS.config

In [3]:
config.multi_device == jax.device_count() > 1
config.model

activation: swish
name: RegressionInceptionNetV1
optimizer: adamw
optimizer_weight_decay: 1.0e-05

In [4]:
jax.devices()

[CudaDevice(id=0)]

In [ ]:
train_ds, _, _ = get_dataset(config)

trainer = TrainerModuleSingle(config=config,
                        model_class=RegressionInceptionNetV1,
                        version=2)

Batch dimensions: [1, 128]
Initializing model with batch shape: (128, 32, 32, 6)


In [ ]:
trainer.train_model(train_ds=train_ds)

2025-06-14 14:07:24.707729: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


KeyboardInterrupt: 